# Setup and Imports

In [1]:
First_Run = False

In [2]:
# Core
import importlib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerBase
import torch
import h5py
import lazy5
from lazy5.inspect import get_datasets, get_attrs_dset
from datetime import datetime
import matplotlib.patches as patches
import time
from dataset_utils import SampleWrapper

# Local modules
import dataset_utils
import plot_utils
import tidytorch_utils as ttu


# Reload during active development
importlib.reload(dataset_utils)
importlib.reload(plot_utils)
importlib.reload(ttu)

# Common imports
from dataset_utils import RamanDataset, voigt_peak, load_h5_file, save_h5_file, ScaleAmpConfig, SampleWrapper



from plot_utils import (
    plot_voigt_fit_res,
    init_plot_context,
    print_condition_stats,
    colored_table_f1,
    _plot_sweep
)

from tidytorch_utils import (
    process_conv_deriv_fit,
    pseudo_voigt,
    denoise_spectrum,
    compute_wavelet_peak,
    precompute_lorentz4_wavelets,
    _match_peaks

)

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("device:", device)



device: cuda


In [3]:

# --------------------
# Axes & scales
# --------------------
x = np.linspace(0, 1824, 1824).astype(np.float32)
widths = np.linspace(0.01, 1, 16)


post_cfg = ScaleAmpConfig(
    score_mode="combined",
    fwhm_mode="hybrid",
    use_quadratic_scale=True,
    fwhm_hybrid_wavelet_weight=0.30,
    use_bias_correction=True,
    return_gamma=True,
    return_sigma=True,
)

sigmas = torch.as_tensor(widths, dtype=torch.float32)   # (n_sigmas,)
gammas = torch.tensor([15.0], dtype=torch.float32)        # (1,) — scalar gamma broadcasts fine
x_gpu  = torch.as_tensor(x, dtype=torch.float32)

x_c  = x_gpu - x_gpu.mean()

sigs = sigmas.view(-1, 1, 1)
gams = gammas.view( 1,-1, 1)
x_bc = x_c.view(   1, 1,-1)

# ── Precomputed constants (shared across all sweep calls) ─────────────────────
_x_dev     = x_gpu.to(device)
_lor4_bank = precompute_lorentz4_wavelets(sigmas.to(device), _x_dev)  # (n_widths, n_pts)

_sigma_t = torch.tensor(3.0,  dtype=torch.float32, device=device)
_gamma_t = torch.tensor(1e-6, dtype=torch.float32, device=device)
_kernel  = compute_wavelet_peak(_sigma_t, _gamma_t, _x_dev)
_kernel  = _kernel / _kernel.sum()
_ker_fft = torch.fft.fft(torch.fft.ifftshift(_kernel))  # (n_pts,)


print(f"Wavelet bank : {_lor4_bank.shape}  on {_lor4_bank.device}")
print(f"Denoise kern : {_ker_fft.shape}    on {_ker_fft.device}")




Wavelet bank : torch.Size([16, 1824])  on cuda:0
Denoise kern : torch.Size([1824])    on cuda:0


# Generate Synthetic Dataset


In [4]:

# Noise / separability grid for synthetic benchmark

BENCH_NOISE_BATCH = [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035,
                0.04, 0.045, 0.05, 0.055, 0.06, 0.065, 0.07, 0.075, 0.08, 0.085, 0.09, 0.095, 0.1]

BENCH_SEP_BATCH   = [(0.6, 0.6), (0.8, 0.8), (1.0, 1.0), (1.2, 1.2), (1.4, 1.4),
                (1.6, 1.6), (1.8, 1.8), (2.0, 2.0), (2.2, 2.2), (2.4, 2.4), (2.6, 2.6), (2.8, 2.8), (3.0, 3.0)]
N_BENCH_BATCH   = 100 


    # BENCH_NOISE_BATCH = [0.0, 0.10,  0.005]
    # BENCH_SEP_BATCH   = [(0.6, 0.6), (0.8, 0.8), (2.0, 2.0)]
    # N_BENCH_BATCH   = 5

In [5]:
#Modify the parameters as needed:

adj = 0.000 
response_threshold=0.001 
amp_threshold = 1e-3  
min_scale_votes=2 
min_spacing_in=7.0  
min_spacing_post = 7.0
max_iter=5000   
tol=1e-5  
scale_preference_fraction=0.8 


CHUNK_SIZE      = 50         # GPU chunk size



In [6]:

# ─── Helpers ─────────────────────────────────────────────────────────────────
def _normalize_noise_levels(levels):
    return [float(n) for n in levels]

def _normalize_sep_levels(levels):
    return [tuple(s) for s in levels]

def _sync_dev(dev):
    """Synchronise GPU so wall-clock timings are accurate."""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()

# ─── Device ──────────────────────────────────────────────────────────────────
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"device: {device}")


# 1) Pre-generate spectra once (CPU)
noise_levels = _normalize_noise_levels(BENCH_NOISE_BATCH)
sep_levels   = _normalize_sep_levels(BENCH_SEP_BATCH)


device: cuda


In [7]:
#Run this once and save it!
if First_Run == True:
    import time as _time
    from joblib import Parallel, delayed

    def _gen_condition(noise_val, sep_val):
        from dataset_utils import RamanDataset, SampleWrapper
        ds = RamanDataset(
            x=x,
            n_peaks=(40, 60),
            widths=widths,
            n_samples=N_BENCH_BATCH,
            seed=123,
            noise_std=noise_val,
            separability_range=sep_val,
            return_both_wavelets=False,
            return_priors=False,
            return_pipeline_estimates=False,
            skip_transforms=True,
        )
        samples = [SampleWrapper(ds[i]) for i in range(N_BENCH_BATCH)]
        labels  = [(noise_val, sep_val)] * N_BENCH_BATCH
        return samples, labels

    conditions = [(n, s) for n in noise_levels for s in sep_levels]
    n_total_samples = len(conditions) * N_BENCH_BATCH
    print(f"Generating {len(conditions)} conditions x {N_BENCH_BATCH} samples = {n_total_samples} total ...")

    samples_bench_batch = []
    condition_labels = []

    _t0 = _time.perf_counter()
    _gen = Parallel(n_jobs=-1, prefer="threads", return_as="generator")(
        delayed(_gen_condition)(n, s) for n, s in conditions
    )
    for _done, (_samples, _labels) in enumerate(_gen, 1):
        samples_bench_batch.extend(_samples)
        condition_labels.extend(_labels)
        _elapsed = _time.perf_counter() - _t0
        _rate = len(samples_bench_batch) / _elapsed
        _eta = (n_total_samples - len(samples_bench_batch)) / max(_rate, 1e-9)
        print(f"  [{_done}/{len(conditions)} conditions | {len(samples_bench_batch)}/{n_total_samples} samples | {_elapsed:.0f}s elapsed | ETA {_eta:.0f}s]")

    print(f"Done in {_time.perf_counter() - _t0:.1f}s")

    print(
        f"Pre-generated {len(samples_bench_batch)} spectra "
        f"({N_BENCH_BATCH} per condition) across "
        f"{len(noise_levels)} noise x {len(sep_levels)} sep conditions"
    )

    spectra_np_batch = np.asarray(
        [np.asarray(ttu._cpu(s.spectrum), dtype=np.float32) for s in samples_bench_batch],
        dtype=np.float32,
    )
    B = spectra_np_batch.shape[0]

    torch.save({"samples_bench_batch": samples_bench_batch, "condition_labels": condition_labels}, f"samples={len(samples_bench_batch)}_bench_batch_{datetime.now().strftime("%d%m%Y")}_{len(noise_levels)}_x_noise_{noise_levels[0]}-{noise_levels[-1]}__{len(sep_levels)}_x_sep_{sep_levels[0]}-{sep_levels[-1]}.pt")


In [8]:
#Run this cell to load the pre-generated spectra (after a restart, or in a new notebook)
if First_Run == False:
    torch.serialization.add_safe_globals([SampleWrapper])

    checkpoint = torch.load('samples=27300_bench_batch_02052026_21_x_noise_0.0-0.1__13_x_sep_(0.6, 0.6)-(3.0, 3.0).pt')
    samples_bench_batch = checkpoint['samples_bench_batch']
    condition_labels = checkpoint['condition_labels']

    spectra_np_batch = np.asarray(
        [np.asarray(ttu._cpu(s.spectrum), dtype=np.float32) for s in samples_bench_batch],
        dtype=np.float32,
    )
    B = spectra_np_batch.shape[0]


# Run the Fitting for a batch

In [9]:
BATCH_SIZE = 500 # number of spectra to process in each batch, depends on memory constraints of your GPU — adjust as needed for larger/smaller cubes or different hardware

MAX_ITER_BATCH = max_iter
TOL_BATCH = tol
AMP_THR_BATCH = amp_threshold
MIN_SPACING_IN_BATCH = min_spacing_in
MIN_SPACING_POST_BATCH = min_spacing_post
RESPONSE_THRESHOLD = response_threshold
MIN_SCALE_VOTES = min_scale_votes
MAX_PEAKS_BATCH = 200

# Aggressive-start controls
AGGR_STEPS = 80 # number of initial steps to use aggressive optimization parameters (higher = more aggressive fitting at start, which can help escape local minima but may cause instability if too high)
AGGR_LR_MULT = 4.0 # learning rate multiplier during aggressive steps (higher = more aggressive updates, which can help escape local minima but may cause instability if too high)
AGGR_CLIP = 3.0 # gradient clipping norm during aggressive steps (lower = more conservative updates, which can improve stability but may slow down convergence if too low)
AGGR_BETA1 = 0.85 # beta1 parameter for Adam during aggressive steps (lower = more momentum, which can help escape local minima but may cause instability if too low)

# Progress checkpoint cadence (in spectra/pixels)
CHECKPOINT_EVERY = 5000 # cadence for progress updates during initial guess construction (lower = more frequent updates but slightly longer runtime due to overhead)

# Progress checkpoint cadence for optimizer iterations
FIT_CHECKPOINT_EVERY = 250 # cadence for progress updates during Adam fitting (lower = more frequent updates but slightly longer runtime due to overhead)


In [10]:
if First_Run:
    name = f'{datetime.now().strftime('%Y%m%d')}_{len(samples_bench_batch)}x_samples_tol={TOL_BATCH}'

else: 
    name = "20260502_27300x_samples_tol=1e-05"

In [11]:
print(name)

20260502_27300x_samples_tol=1e-05


In [12]:
# ── Resume from checkpoint (set RESUME=True and run before the main loop) ────
# The checkpoint file is keyed to `name`, so it will match the current run.

RESUME = True

resume_from    = 0
_resume_params = None
_resume_model  = None
_resume_spec_d = None
_resume_stats  = None

if RESUME:
    _ckpt_path = f"synthetic_fit_{name}_checkpoint.pt"
    if not os.path.exists(_ckpt_path):
        raise FileNotFoundError(f"No checkpoint found at {_ckpt_path}")

    print(f"Loading checkpoint: {_ckpt_path}")
    _ckpt = torch.load(_ckpt_path, weights_only=False)

    resume_from    = int(_ckpt['batches_completed'])
    _resume_params = _ckpt['params_all']   # (resume_from, target_cols)
    _resume_model  = _ckpt['model_all']    # (resume_from, N_PTS)
    _resume_spec_d = _ckpt['spec_d']       # (resume_from, N_PTS)
    _resume_stats  = _ckpt['all_stats']    # list of length resume_from

    print(f"  Resuming from spectrum {resume_from} / {_ckpt['n_total']}")
    print(f"  F1 so far: mean={_ckpt['f1_vals'].mean():.3f}")

Loading checkpoint: synthetic_fit_20260502_27300x_samples_tol=1e-05_checkpoint.pt
  Resuming from spectrum 12500 / 27300
  F1 so far: mean=0.775


In [ ]:
ttu.init_sweep_context(x_gpu, sigmas, gammas, device, widths)
init_plot_context(x, widths)

if spectra_np_batch.ndim != 2:
    raise ValueError(
        f"Expected spectra_np_batch with shape (N, n_pts), got {spectra_np_batch.shape}"
    )

N_TOTAL, N_PTS = spectra_np_batch.shape
print(f"Input dataset shape: {spectra_np_batch.shape}")

# Timing/stat containers
batch_total_times = []
batch_fit_times = []
tbp = []
tfp = []
p_fit = []

# Pre-allocate output arrays (avoids holding all batches in memory until concat)
target_cols = MAX_PEAKS_BATCH * 4
params_all  = np.zeros((N_TOTAL, target_cols), dtype=np.float32)
model_all   = np.zeros((N_TOTAL, N_PTS),       dtype=np.float32)
spec_d_all  = np.zeros((N_TOTAL, N_PTS),       dtype=np.float32)
all_stats   = [None] * N_TOTAL

x_np          = np.asarray(x, dtype=np.float32)
CTR_TOL_BATCH = min_spacing_in
checkpoint_path = f"synthetic_fit_{name}_checkpoint.pt"

# Restore checkpoint data if resuming
if resume_from > 0:
    params_all[:resume_from] = _resume_params
    model_all[:resume_from]  = _resume_model
    spec_d_all[:resume_from] = _resume_spec_d
    all_stats[:resume_from]  = _resume_stats
    print(f"Restored {resume_from} spectra from checkpoint — starting loop at {resume_from}")

_sync_dev(ttu._ctx_device)
t_init = time.perf_counter()

with torch.no_grad():
    if ttu._ctx_device.type == "cuda":
        torch.cuda.empty_cache()

for start in range(resume_from, N_TOTAL, BATCH_SIZE):
    end = min(start + BATCH_SIZE, N_TOTAL)
    spectra_np = spectra_np_batch[start:end]
    B = spectra_np.shape[0]
    print(f"\n[batch {start}:{end}] spectra: {B} x {spectra_np.shape[1]}")

    spectra_gpu = torch.as_tensor(spectra_np, dtype=torch.float32, device=ttu._ctx_device)

    with torch.no_grad():
        _ = ttu._denoise_batch(spectra_gpu[:1])

    _sync_dev(ttu._ctx_device)
    t0 = time.perf_counter()

    # 1) Denoise + transform + vote
    spec_d = ttu._denoise_batch(spectra_gpu)

    resp = ttu._lor4_transform_batch(spec_d)
    _, n_widths, n_pts = resp.shape

    flat_masks = ttu.find_peaks_derivative_mask_batch(
        ttu._x_dev, resp.reshape(B * n_widths, n_pts), min_height=0.0
    )
    masks = flat_masks.reshape(B, n_widths, n_pts)
    vote_count = masks.long().sum(dim=1)
    max_resp = resp.max(dim=1).values
    peak_masks = (vote_count >= MIN_SCALE_VOTES) & (max_resp > RESPONSE_THRESHOLD)

    # 2) Build padded p0 + gradient mask
    npq_max = MAX_PEAKS_BATCH * 4
    p0_flat_list = []
    n_real_peaks = []

    t_init0 = time.perf_counter()
    for i in range(B):
        p0 = ttu.build_initial_guesses_from_derivative_mask(
            resp[i].unsqueeze(1),
            ttu._ctx_sigmas.to(ttu._ctx_device),
            ttu._ctx_gammas[:1].to(ttu._ctx_device),
            ttu._x_dev,
            spec_d[i],
            peak_masks[i],
            max_peaks=MAX_PEAKS_BATCH,
            min_spacing=MIN_SPACING_IN_BATCH,
            scale_preference_fraction=0.008,
        )
        p0_flat = p0[:npq_max]
        p0_flat_list.append(p0_flat)
        n_peaks_i = int((p0_flat[0::4] > 0).sum().item())
        n_real_peaks.append(min(n_peaks_i, MAX_PEAKS_BATCH))

        done = i + 1
        if (done % CHECKPOINT_EVERY == 0) or (done == B):
            elapsed = time.perf_counter() - t_init0
            px_per_sec = done / max(elapsed, 1e-9)
            eta = (B - done) / max(px_per_sec, 1e-9)
            print(f"[checkpoint] initialized {done}/{B} spectra")
            print(f"({100.0 * done / B:.1f}%) | elapsed {elapsed:.1f}s | ETA {eta:.1f}s")

    npq = MAX_PEAKS_BATCH * 4
    p0_batch = torch.zeros(B, npq, device=ttu._ctx_device)
    grad_mask = torch.zeros(B, npq, dtype=torch.float32, device=ttu._ctx_device)

    for i, (p0_flat, n_peaks_i) in enumerate(zip(p0_flat_list, n_real_peaks)):
        n_take = min(p0_flat.numel(), npq)
        p0_batch[i, :n_take] = p0_flat[:n_take]
        grad_mask[i, : min(n_peaks_i * 4, npq)] = 1.0

    # 3) Batched Adam fit
    _sync_dev(ttu._ctx_device)
    t_fit0 = time.perf_counter()
    params_batch = ttu._fit_batch_adam(
        spec_d,
        ttu._x_dev,
        p0_batch,
        grad_mask,
        max_iter=MAX_ITER_BATCH,
        tol=TOL_BATCH,
        aggressive_start_steps=AGGR_STEPS,
        aggressive_lr_mult=AGGR_LR_MULT,
        aggressive_clip_norm=AGGR_CLIP,
        aggressive_beta1=AGGR_BETA1,
        progress_every=FIT_CHECKPOINT_EVERY,
        progress_prefix="fit",
    )
    _sync_dev(ttu._ctx_device)
    fit_only_sec = time.perf_counter() - t_fit0

    # 4) Post-process fitted peaks
    params_batch_metrics = ttu._prune_peaks_batch(params_batch, amp_threshold=AMP_THR_BATCH)
    params_batch_metrics = ttu._deduplicate_peaks_batch(params_batch_metrics, MIN_SPACING_POST_BATCH)

    cur_cols = params_batch_metrics.shape[1]
    if cur_cols < target_cols:
        pad = torch.zeros(B, target_cols - cur_cols,
                          dtype=params_batch_metrics.dtype,
                          device=params_batch_metrics.device)
        params_batch_metrics = torch.cat([params_batch_metrics, pad], dim=1)
    elif cur_cols > target_cols:
        params_batch_metrics = params_batch_metrics[:, :target_cols]

    params_all[start:end] = params_batch_metrics.cpu().numpy().astype(np.float32)

    # 5) Recover fitted spectra
    with torch.no_grad():
        model_all[start:end] = ttu._compute_model_batch(params_batch, ttu._x_dev).cpu().numpy().astype(np.float32)

    spec_d_all[start:end] = spec_d.cpu().numpy().astype(np.float32)

    _sync_dev(ttu._ctx_device)
    total_sec = time.perf_counter() - t0

    n_fit_peaks = (params_batch.reshape(B, -1, 4)[:, :, 0] > AMP_THR_BATCH).sum(dim=1)
    print("\nBatch fit summary:")
    print(f"  total time: {total_sec:.3f}s  -> {total_sec / B:.4f}s per spectrum")
    print(f"  fit-only : {fit_only_sec:.3f}s  -> {fit_only_sec / B:.4f}s per spectrum")
    print(
        f"  fitted peaks/spectrum: mean={n_fit_peaks.float().mean().item():.1f}  "
        f"median={n_fit_peaks.float().median().item():.1f}"
    )

    batch_total_times.append(total_sec)
    batch_fit_times.append(fit_only_sec)
    tbp.append(total_sec / B)
    tfp.append(fit_only_sec / B)
    p_fit.append(n_fit_peaks.float().mean().item())

    # 6) Peak matching for this batch
    for i_global in range(start, end):
        s = samples_bench_batch[i_global]
        all_stats[i_global] = ttu._match_peaks(
            s.centers, s.amplitudes, s.sigmas, s.gammas,
            params_all[i_global],
            tolerance=CTR_TOL_BATCH,
            amp_threshold=AMP_THR_BATCH,
            x_arr=x_np,
        )

    # 7) Incremental checkpoint save
    completed_stats = all_stats[:end]
    f1_so_far   = np.asarray([s['f1']        for s in completed_stats], dtype=np.float64)
    prec_so_far = np.asarray([s['precision'] for s in completed_stats], dtype=np.float64)
    rec_so_far  = np.asarray([s['recall']    for s in completed_stats], dtype=np.float64)
    torch.save({
        'params_all':        params_all[:end],
        'model_all':         model_all[:end],
        'spectra_np':        spectra_np_batch[:end].astype(np.float32),
        'spec_d':            spec_d_all[:end],
        'condition_labels':  condition_labels[:end],
        'x_axis':            x_np,
        'f1_vals':           f1_so_far,
        'prec_vals':         prec_so_far,
        'rec_vals':          rec_so_far,
        'all_stats':         completed_stats,
        'batches_completed': end,
        'n_total':           N_TOTAL,
    }, checkpoint_path)
    print(f"  [checkpoint saved: {end}/{N_TOTAL} spectra -> {checkpoint_path}]")

    del params_batch, p0_batch, flat_masks, masks, resp
    del spectra_gpu, spec_d, grad_mask, peak_masks, vote_count
    del p0_flat_list, n_real_peaks

_sync_dev(ttu._ctx_device)
t_final = time.perf_counter()

f1_vals   = np.asarray([s['f1']        for s in all_stats], dtype=np.float64)
prec_vals = np.asarray([s['precision'] for s in all_stats], dtype=np.float64)
rec_vals  = np.asarray([s['recall']    for s in all_stats], dtype=np.float64)

print(f"\nCompleted {N_TOTAL} spectra in {t_final - t_init:.3f}s")
print(f"params_all shape: {params_all.shape}")
print(f"model_all shape:  {model_all.shape}")
print(f"\nPeak matching (tol={CTR_TOL_BATCH}):")
print(f"  F1        : mean={f1_vals.mean():.3f}  std={f1_vals.std():.3f}")
print(f"  Precision : mean={prec_vals.mean():.3f}  std={prec_vals.std():.3f}")
print(f"  Recall    : mean={rec_vals.mean():.3f}  std={rec_vals.std():.3f}")

results_filename = f"synthetic_fit_{name}.pt"
torch.save({
    'params_all':       params_all,
    'model_all':        model_all,
    'spectra_np':       spectra_np_batch.astype(np.float32),
    'spec_d':           spec_d_all,
    'condition_labels': condition_labels,
    'x_axis':           x_np,
    'f1_vals':          f1_vals,
    'prec_vals':        prec_vals,
    'rec_vals':         rec_vals,
    'all_stats':        all_stats,
}, results_filename)
print(f"Saved: {results_filename}")

print("\nTiming summary:")
print(f"  average batch total time: {np.mean(batch_total_times):.3f}s +/- {np.std(batch_total_times):.3f}s")
print(f"  average batch fit time  : {np.mean(batch_fit_times):.3f}s +/- {np.std(batch_fit_times):.3f}s")
print(f"  average total time/spectrum: {np.mean(tbp):.4f}s +/- {np.std(tbp):.4f}s")
print(f"  average fit time/spectrum  : {np.mean(tfp):.4f}s +/- {np.std(tfp):.4f}s")
print(f"  average fitted peaks/spectrum: {np.mean(p_fit):.1f} +/- {np.std(p_fit):.1f}")

Sweep context initialised on cuda
  Wavelet bank : torch.Size([16, 1824])
  Denoise kern : torch.Size([1824])
Input dataset shape: (27300, 1824)
Restored 12500 spectra from checkpoint — starting loop at 12500

[batch 12500:13000] spectra: 500 x 1824
[checkpoint] initialized 500/500 spectra
(100.0%) | elapsed 1.4s | ETA 0.0s
[fit] iter 250/5000 (5.0%) | loss/pix=2.3261e+00 | lr=9.99e-03 | elapsed 24.7s | ETA 469.6s
[fit] iter 500/5000 (10.0%) | loss/pix=1.4932e+00 | lr=9.89e-03 | elapsed 53.1s | ETA 477.8s
[fit] iter 750/5000 (15.0%) | loss/pix=9.9361e-01 | lr=9.66e-03 | elapsed 82.6s | ETA 468.3s
[fit] iter 1000/5000 (20.0%) | loss/pix=5.8230e-01 | lr=9.31e-03 | elapsed 109.2s | ETA 436.6s
[fit] iter 1250/5000 (25.0%) | loss/pix=2.9965e-01 | lr=8.84e-03 | elapsed 136.2s | ETA 408.7s
[fit] iter 1500/5000 (30.0%) | loss/pix=1.8202e-01 | lr=8.27e-03 | elapsed 161.8s | ETA 377.6s
[fit] iter 1750/5000 (35.0%) | loss/pix=1.5103e-01 | lr=7.61e-03 | elapsed 187.9s | ETA 348.9s
[fit] iter 2000/

In [ ]:
# 6) Match fitted peaks against ground-truth priors
# _match_peaks is pure numpy — pass CPU arrays directly, no GPU round-trips
x_np = np.asarray(x, dtype=np.float32)
CTR_TOL_BATCH = min_spacing_in  # centre tolerance for a true-positive match
all_stats = []
for i, s in enumerate(samples_bench_batch):
    stats = ttu._match_peaks(
        s.centers, s.amplitudes, s.sigmas, s.gammas,
        params_all[i],
        tolerance=CTR_TOL_BATCH,
        amp_threshold=AMP_THR_BATCH,
        x_arr=x_np,
    )
    all_stats.append(stats)

f1_vals   = np.asarray([s['f1']        for s in all_stats], dtype=np.float64)
prec_vals = np.asarray([s['precision'] for s in all_stats], dtype=np.float64)
rec_vals  = np.asarray([s['recall']    for s in all_stats], dtype=np.float64)
print(f"\nPeak matching (tol={CTR_TOL_BATCH}):")
print(f"  F1        : mean={f1_vals.mean():.3f}  std={f1_vals.std():.3f}")
print(f"  Precision : mean={prec_vals.mean():.3f}  std={prec_vals.std():.3f}")
print(f"  Recall    : mean={rec_vals.mean():.3f}  std={rec_vals.std():.3f}")

In [ ]:
noise_levels = _normalize_noise_levels(BENCH_NOISE_BATCH)
sep_levels   = _normalize_sep_levels(BENCH_SEP_BATCH)

# condition_labels = []

# for noise_idx, noise_val in enumerate(noise_levels):
#     for sep_idx, sep_val in enumerate(sep_levels):
#         condition_labels.extend([(noise_val, sep_val)] * N_BENCH_BATCH)

print_condition_stats(condition_labels, all_stats)
colored_table_f1(condition_labels, all_stats)

In [ ]:
# ── Recreate Noise-Level and Peak-Overlap style sweep plots from benchmark outputs ──
importlib.reload(plot_utils)
params_np = params_all
spec_d = spec_d_all

from plot_utils import init_plot_context, _plot_sweep
init_plot_context(x, widths)

n_cond = len(condition_labels)
n_stats = len(all_stats)
if n_cond == 0 or n_stats == 0:
    raise ValueError("Empty benchmark outputs. Re-run the matrix benchmark cell.")

if n_stats == n_cond:
    labels_aligned = condition_labels
    repeat_factor = 1
elif (n_stats % n_cond) == 0:
    repeat_factor = n_stats // n_cond
    labels_aligned = condition_labels * repeat_factor
    print(f"Detected {repeat_factor} benchmark repeats; aggregating all repeats in sweep plots.")
else:
    raise ValueError(
        f"Length mismatch: condition_labels={n_cond} vs all_stats_batch={n_stats}. "
        "Re-run the matrix benchmark cell to refresh outputs."
    )

# Group full stats dicts by exact condition (noise, sep)
stats_by_cond = {}
for (noise, sep), st in zip(labels_aligned, all_stats):
    noise_f = float(noise)
    sep_t = (float(sep[0]), float(sep[1]))
    stats_by_cond.setdefault((noise_f, sep_t), []).append(st)

noise_levels = sorted({k[0] for k in stats_by_cond.keys()})
sep_levels = sorted({k[1] for k in stats_by_cond.keys()}, key=lambda t: (t[0], t[1]))

sep_fix = [(2.0, 2.0)]
noise_fix = [0.0]
print(f"Preparing sweep-style plots for {len(noise_levels)} noise levels and {len(sep_levels)} separability ranges.")

# 1) Noise-level style plots (one figure per fixed separability range)
for sep_fixed in sep_fix:
    all_results_noise = {}
    for noise_val in noise_levels:
        all_results_noise[noise_val] = stats_by_cond.get((noise_val, sep_fixed), [])

    if any(len(v) > 0 for v in all_results_noise.values()):
        print(f"Noise-style plot @ sep={sep_fixed}")
        _plot_sweep(
            param_name='noise_std',
            param_values=noise_levels,
            all_results=all_results_noise,
            samples=samples_bench_batch,
            params=params_np,
            spec_d_batch=spec_d,
            precomp_stats=all_stats,
            condition_labels=condition_labels,
            example_idx=(0, -1),
            example_fixed_val=sep_fixed,
        )

# 2) Peak-overlap style plots (one figure per fixed noise level)
for noise_fixed in noise_fix:
    all_results_overlap = {}
    for sep_val in sep_levels:
        all_results_overlap[str(sep_val)] = stats_by_cond.get((noise_fixed, sep_val), [])

    if any(len(v) > 0 for v in all_results_overlap.values()):
        print(f"Overlap-style plot @ noise={noise_fixed}")
        _plot_sweep(
            param_name='separability',
            param_values=sep_levels,
            all_results=all_results_overlap,
            samples=samples_bench_batch,
            params=params_np,
            spec_d_batch=spec_d,
            precomp_stats=all_stats,
            condition_labels=condition_labels,
            example_idx=(0, -1),
            example_fixed_val=noise_fixed,
        )

In [ ]:
def per_peak_records_from_batch(all_stats_batch, samples_bench_batch, condition_labels,
                                 amp_thr=1e-2):
    """
    Build per-GT-peak scatter records from a pre-computed all_stats_batch.
    """
    records = []
    for stats, sample, (noise_std, _sep) in zip(all_stats_batch, samples_bench_batch, condition_labels):
        gt_c = np.asarray((sample.centers))
        gt_a = np.asarray((sample.amplitudes))
        gt_s = np.asarray((sample.sigmas))

        valid = gt_a > amp_thr
        gt_c, gt_a, gt_s = gt_c[valid], gt_a[valid], gt_s[valid]
        if len(gt_a) == 0:
            continue

        n_tp = int(stats['tp'])
        amp_order = np.argsort(gt_a)[::-1]
        detected = np.zeros(len(gt_a), dtype=float)
        detected[amp_order[:n_tp]] = 1.0

        fwhm = 2.355 * gt_s + 1e-9

        for k in range(len(gt_a)):
            other_c = np.delete(gt_c, k)
            sep_metric = (float(np.min(np.abs(other_c - gt_c[k]))) / fwhm[k]
                          if len(other_c) > 0 else 10.0)
            noise_over_amp = noise_std / float(gt_a[k]) if noise_std > 0 else 0.0
            records.append(dict(
                noise_over_amp=noise_over_amp,
                separability=sep_metric,
                amplitude=float(gt_a[k]),
                noise_std=noise_std,
                detected=detected[k],
                spectrum_f1=float(stats['f1']),
            ))
    return records

In [ ]:
# ── Detection-rate contour + density: noise/amp × separability ───────────────
import seaborn as sns
from scipy.ndimage import gaussian_filter
from scipy.stats import binned_statistic_2d



peak_records = per_peak_records_from_batch(all_stats, samples_bench_batch, condition_labels)
df_pk = pd.DataFrame(peak_records)
det_rate = df_pk['detected'].mean()
print(f"Total GT peaks: {len(df_pk)}  |  Overall detection rate: {det_rate:.1%}")


# Drop noise_over_amp == 0 (can't log-scale noiseless peaks)
df_plot = df_pk[df_pk['noise_over_amp'] > 0].copy()

log_x = np.log10(df_plot['noise_over_amp'].values)
log_y = np.log10(df_plot['separability'].values)
z     = df_plot['detected'].values
if log_x.size == 0 or log_y.size == 0:
    print("Error: One of the input arrays is empty!")
else:
    # Bin detection rate onto a 2D grid in log-space
    N_BINS = 40
    det_rate_grid, xe, ye, _ = binned_statistic_2d(
        log_x, log_y, z, statistic='mean', bins=N_BINS
    )
    count_grid, _, _, _      = binned_statistic_2d(
        log_x, log_y, z, statistic='count', bins=N_BINS
    )

# Smooth; mask bins with no data
det_smooth = gaussian_filter(np.nan_to_num(det_rate_grid, nan=0.5), sigma=1.5)
det_smooth[count_grid == 0] = np.nan

xc = 10 ** (0.5 * (xe[:-1] + xe[1:]))
yc = 10 ** (0.5 * (ye[:-1] + ye[1:]))
XX, YY = np.meshgrid(xc, yc)

fig, ax = plt.subplots(figsize=(9, 6), dpi=120)

# Filled contour: detection rate
cf = ax.contourf(XX, YY, det_smooth.T, levels=np.linspace(0, 1, 40),
                 cmap='managua', alpha=0.9)
plt.colorbar(cf, ax=ax, label='Detection Rate')

# ── Density contour range filters (spectral characteristics) ─────────────────
# noise_std:    spectrum-level noise σ added during simulation
# separability: per-peak min-dist/FWHM (spectral condition proxy)
# Set to None for no limit on that side.
noise_std_min, noise_std_max = 0.00, 0.015   # noise_std  (e.g. 0.02, 0.10)
sep_min,       sep_max       = 1.0, 1.8   # separability (e.g. 0.8, 2.0)

mask = pd.Series(True, index=df_plot.index)
if noise_std_min is not None: mask &= df_plot['noise_std'] >= noise_std_min
if noise_std_max is not None: mask &= df_plot['noise_std'] <= noise_std_max
if sep_min       is not None: mask &= df_plot['separability'] >= sep_min
if sep_max       is not None: mask &= df_plot['separability'] <= sep_max
df_density = df_plot[mask]

# Contour lines: data density
sns.kdeplot(data=df_density, x='noise_over_amp', y='separability',
            ax=ax, log_scale=True,
            levels=5, color='k', linewidths=0.8, alpha=0.45,
            label='Data density')

# Clamp axes to the heatmap grid extent so KDE lines don't stray outside
ax.set_xlim(10**xe[1], 10**xe[-12])
ax.set_ylim(10**ye[3], 10**ye[-14])

ax.axhline(1.0, color='white', lw=1.2, ls='--', alpha=1, label='1× FWHM')
ax.axhline(2.0, color='white', lw=1.2, ls=':',  alpha=1, label='2× FWHM')
ax.set_xscale('log')
ax.set_yscale('log')
from matplotlib.ticker import ScalarFormatter

for axis in [ax.xaxis, ax.yaxis]:
    axis.set_major_formatter(ScalarFormatter())
    axis.get_major_formatter().set_scientific(False)

ax.set_xlabel('Noise / Peak Amplitude', fontsize=12)
ax.set_ylabel('Peak Separability (min dist / FWHM)', fontsize=12)
ax.set_title('Per-peak Detection Rate', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('per_peak_contour.png', dpi=300, bbox_inches='tight')
plt.show()
